# ot3.ipynb

동시 판정 통합 실행 노트북입니다.

우선순위를 적용하지 않습니다.

1. **bd4 행동 판정**: 모든 영상에 대해 `fight / stalking / other`를 판정합니다.
2. **cf6 이륜차 계열 판정**: 모든 영상에 대해 이륜차/자전거/오토바이 + 인도 주행/헬멧 미착용을 판정합니다.
3. **동시 통합 결과**: 한 영상에 대해 `bd4_result`와 `cf6_result`를 모두 보존하고, `combined_tags`에 동시에 기록합니다.

입력 영상 루트:

```python
C:\Users\우성한\Sample (1)\Sample\01.원천데이터
```



In [1]:
# Cell 1. 전체 경로/설정
from pathlib import Path
import os
import json
import math
import time
import gc
import tempfile
from collections import Counter, defaultdict

# ===============================
# 핵심 경로
# ===============================
VIDEO_ROOT = Path(r"C:\Users\우성한\Sample (1)\Sample\01.원천데이터")
OUTPUT_ROOT = Path(r"C:\Users\우성한\outputs\prediction")

CF2_DIR = OUTPUT_ROOT / "cf2"
OT3_DIR = OUTPUT_ROOT / "ot3"
OT_DIR = OT3_DIR
OT_CF6_DIR = OT3_DIR / "cf6_parallel"
OT_BD4_DIR = OT3_DIR / "bd4_parallel"
OT_DEBUG_DIR = OT3_DIR / "debug"
HELMET_CROP_DIR = OT_DEBUG_DIR / "helmet_crops"

for d in [CF2_DIR, OT_DIR, OT_CF6_DIR, OT_BD4_DIR, OT_DEBUG_DIR, HELMET_CROP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ===============================
# 처리 제한: 테스트 시 숫자로 바꾸고, 전체 처리 시 None
# ===============================
MAX_VIDEOS = None
MAX_JSON_FILES = None
MAX_FRAMES_PER_VIDEO = None
# MAX_VIDEOS = 2
# MAX_JSON_FILES = 2
# MAX_FRAMES_PER_VIDEO = 120

# ===============================
# CF2 / YOLO 설정
# ===============================
VIDEO_EXTS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv", ".mpeg", ".mpg", ".m4v",
    ".MP4", ".AVI", ".MOV", ".MKV", ".WMV", ".MPEG", ".MPG", ".M4V"
}
CF2_FRAME_INTERVAL = 15
YOLO_MODEL_PATH = "yolov8n.pt"

# ===============================
# cf6 설정: 이륜차/인도주행/헬멧
# ===============================
SEG_MODEL_ID_GPU = "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"
SEG_MODEL_ID_CPU = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"

USE_ROBOFLOW_HOSTED_INFERENCE = False
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
ROBOFLOW_MODEL_ID = "bike-helmet-detection-2vdjo/1"

# 로컬 헬멧 모델이 있으면 여기에 경로 입력
# 예: HELMET_MODEL_PATH = r"C:\Users\우성한\models\bike_helmet_best.pt"
HELMET_MODEL_PATH = None

PERSON_CLASSES = {"person", "rider"}
VEHICLE_CLASSES = {"bicycle", "motorcycle"}
MIN_PERSON_CONF = 0.10
MIN_VEHICLE_CONF = 0.10
MAX_CENTER_DISTANCE = 280
MIN_OVERLAP_RATIO = 0.015

ROAD_ID = 0
SIDEWALK_ID = 1
MASK_PATCH_RADIUS = 16
DILATE_KERNEL_SIZE = 9
BOTTOM_BAND_X_MARGIN = 0.08
SIDEWALK_SCORE_THRESHOLD = 0.18
ROAD_SCORE_THRESHOLD = 0.18
SIDEWALK_JUDGE_VERSION = "ot_cf6_v2_vehicle_union_bbox"

HELMET_CONF = 0.15
SAVE_HELMET_CROPS = True
MAX_SAVED_HELMET_CROPS = 300

USE_AMP_ON_CUDA = True
CLEAR_CUDA_CACHE_EVERY_N_VIDEOS = 3
SAVE_PARTIAL_EVERY_N_VIDEOS = 5

# ===============================
# 최종 통합 우선순위
# ===============================
CF6_EVENT_STATUSES = {"sidewalk_riding", "sidewalk_candidate", "no_helmet"}

print("VIDEO_ROOT exists:", VIDEO_ROOT.exists(), VIDEO_ROOT)
print("CF2_DIR:", CF2_DIR)
print("OT3_DIR:", OT3_DIR)
print("ROBOFLOW_API_KEY set:", bool(ROBOFLOW_API_KEY))

if not VIDEO_ROOT.exists():
    raise FileNotFoundError(f"원본 영상 루트를 찾지 못했습니다: {VIDEO_ROOT}")




VIDEO_ROOT exists: True C:\Users\우성한\Sample (1)\Sample\01.원천데이터
CF2_DIR: C:\Users\우성한\outputs\prediction\cf2
OT3_DIR: C:\Users\우성한\outputs\prediction\ot3
ROBOFLOW_API_KEY set: False


In [2]:
# Cell 2. 의존성 확인 및 모델 import
missing = []

try:
    import cv2
    print("cv2 OK")
except Exception as e:
    missing.append("opencv-python")
    print("cv2 import failed:", repr(e))

try:
    import numpy as np
    print("numpy OK")
except Exception as e:
    missing.append("numpy")
    print("numpy import failed:", repr(e))

try:
    import torch
    print("torch OK:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
        torch.backends.cudnn.benchmark = True
except Exception as e:
    missing.append("torch")
    print("torch import failed:", repr(e))

try:
    from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
    print("transformers OK")
except Exception as e:
    missing.append("transformers")
    print("transformers import failed:", repr(e))

try:
    from ultralytics import YOLO
    print("ultralytics OK")
except Exception as e:
    YOLO = None
    missing.append("ultralytics")
    print("ultralytics import failed:", repr(e))

try:
    from PIL import Image
    print("PIL OK")
except Exception as e:
    missing.append("pillow")
    print("PIL import failed:", repr(e))

if missing:
    print("\n설치 필요 후보:", sorted(set(missing)))
    print("예: pip install opencv-python numpy torch transformers ultralytics pillow inference-sdk")




cv2 OK
numpy OK
torch OK: 2.6.0+cu126
cuda available: True
gpu: NVIDIA GeForce RTX 3060
transformers OK
ultralytics OK
PIL OK


In [3]:
# Cell 3. 원천데이터 전체 영상 탐색
video_paths = [p for p in VIDEO_ROOT.rglob("*") if p.is_file() and p.suffix in VIDEO_EXTS]
video_paths = sorted(video_paths)

if MAX_VIDEOS is not None:
    video_paths = video_paths[:MAX_VIDEOS]

print("찾은 영상 수:", len(video_paths))
for p in video_paths[:20]:
    print(p)

if len(video_paths) == 0:
    raise RuntimeError("원천데이터 경로 안에서 영상 파일을 찾지 못했습니다.")




찾은 영상 수: 22
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\전동킥보드 인도 주행\es_e3768\es_e3768_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데

In [4]:
# Cell 4. CF2 생성/확인
if YOLO is None:
    raise RuntimeError("ultralytics YOLO를 사용할 수 없습니다.")

cf2_yolo_model = YOLO(YOLO_MODEL_PATH)
print("YOLO model loaded:", YOLO_MODEL_PATH)


def safe_stem_from_path(path: Path, root: Path):
    rel = path.relative_to(root)
    return "__".join(rel.with_suffix("").parts)


def create_cf2_from_video(video_path, output_json_path, frame_interval=15):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")

    frames = []
    frame_idx = 0
    sampled_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            result = cf2_yolo_model(frame, verbose=False)[0]
            objects = []
            classes = {}

            if result.boxes is not None:
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    cls_name = cf2_yolo_model.names[cls_id]
                    conf = float(box.conf[0])
                    bbox = [round(float(v), 2) for v in box.xyxy[0].tolist()]
                    objects.append({"class": cls_name, "confidence": round(conf, 4), "bbox": bbox})
                    classes[cls_name] = classes.get(cls_name, 0) + 1

            frames.append({
                "frame_id": frame_idx,
                "sampled_id": sampled_idx,
                "object_count": len(objects),
                "classes": classes,
                "objects": objects,
            })
            sampled_idx += 1

        frame_idx += 1

    cap.release()

    summary_classes = {}
    for fr in frames:
        for cls_name, count in fr["classes"].items():
            summary_classes[cls_name] = summary_classes.get(cls_name, 0) + count

    data = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "relative_video_path": str(video_path.relative_to(VIDEO_ROOT)),
        "summary": {
            "total_frames_read": frame_idx,
            "sampled_frames": len(frames),
            "classes": summary_classes,
        },
        "frames": frames,
    }

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return data


created = 0
used_existing = 0
failed_cf2 = []

from tqdm import tqdm
for video_path in tqdm(video_paths):
    stem = safe_stem_from_path(video_path, VIDEO_ROOT)
    output_json_path = CF2_DIR / f"{stem}.json"

    if output_json_path.exists():
        used_existing += 1
        continue

    try:
        create_cf2_from_video(video_path, output_json_path, frame_interval=CF2_FRAME_INTERVAL)
        created += 1
    except Exception as e:
        failed_cf2.append({"video": str(video_path), "error": repr(e)})

print("\n================================")
print("CF2 생성/확인 완료")
print("전체 영상:", len(video_paths))
print("생성:", created)
print("기존 사용:", used_existing)
print("실패:", len(failed_cf2))
print("================================")
if failed_cf2:
    for item in failed_cf2[:20]:
        print(item)




YOLO model loaded: yolov8n.pt


100%|████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 3590.45it/s]


CF2 생성/확인 완료
전체 영상: 22
생성: 0
기존 사용: 22
실패: 0


In [5]:
# Cell 5. CF2 JSON / 원본 영상 인덱싱
json_files = sorted(CF2_DIR.glob("*.json"))
if MAX_JSON_FILES is not None:
    json_files = json_files[:MAX_JSON_FILES]

print("CF2 json files:", len(json_files))
print([p.name for p in json_files[:10]])

if not json_files:
    raise RuntimeError("CF2 JSON 파일이 없습니다. Cell 4가 정상 실행되었는지 확인하세요.")

print("\n원본 영상 인덱싱 중...")
t0 = time.time()
video_index = {}
for p in VIDEO_ROOT.rglob("*"):
    if p.is_file() and p.suffix in VIDEO_EXTS:
        if p.name not in video_index:
            video_index[p.name] = p
print("indexed videos:", len(video_index), "elapsed_sec:", round(time.time() - t0, 2))


def find_video_path(video_name: str) -> Path:
    p = video_index.get(video_name)
    if p is not None:
        return p
    stem = Path(video_name).stem
    candidates = [v for k, v in video_index.items() if Path(k).stem == stem]
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f"video not found for {video_name}")

with open(json_files[0], encoding="utf-8") as f:
    sample = json.load(f)
print("sample file:", json_files[0].name)
print("sample video_name:", sample.get("video_name"))
print("sample frame count:", len(sample.get("frames", [])))




CF2 json files: 22
['스토킹__특정 구역 내 지속 배회__lo_e1281__lo_e1281_c1.json', '스토킹__특정 구역 내 지속 배회__lo_e1281__lo_e1281_c2.json', '스토킹__특정 인물을 뒤따라가며 배회__fp_e1520__fp_e1520_c1.json', '스토킹__특정 인물을 뒤따라가며 배회__fp_e1520__fp_e1520_c2.json', '싸움__신체적 충돌을 동반한 싸움__ph_e2063__ph_e2063_c1.json', '싸움__신체적 충돌을 동반한 싸움__ph_e2063__ph_e2063_c2.json', '이륜 이동수단 인도 주행__오토바이 인도 주행__ms_e3500__ms_e3500_c1.json', '이륜 이동수단 인도 주행__오토바이 인도 주행__ms_e3500__ms_e3500_c2.json', '이륜 이동수단 인도 주행__자전거 인도 주행__bs_e4387__bs_e4387_c1.json', '이륜 이동수단 인도 주행__자전거 인도 주행__bs_e4387__bs_e4387_c2.json']

원본 영상 인덱싱 중...
indexed videos: 22 elapsed_sec: 0.02
sample file: 스토킹__특정 구역 내 지속 배회__lo_e1281__lo_e1281_c1.json
sample video_name: lo_e1281_c1.mp4
sample frame count: 61


In [6]:
# Cell 6. cf6 bbox / rider-vehicle 매칭 함수

def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(0, x2 - x1) * max(0, y2 - y1)


def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def bottom_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, y2)


def intersection_area(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    x1 = max(ax1, bx1)
    y1 = max(ay1, by1)
    x2 = min(ax2, bx2)
    y2 = min(ay2, by2)
    return max(0, x2 - x1) * max(0, y2 - y1)


def dist(a, b):
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


def extract_objects(frame):
    persons = []
    vehicles = []
    for obj in frame.get("objects", []):
        cls = obj.get("class")
        conf = float(obj.get("confidence", 0.0))
        bbox = obj.get("bbox")
        if not bbox or len(bbox) != 4:
            continue
        clean = {
            "class": cls,
            "confidence": round(conf, 4),
            "bbox": [int(round(v)) for v in bbox],
        }
        if cls in PERSON_CLASSES and conf >= MIN_PERSON_CONF:
            persons.append(clean)
        elif cls in VEHICLE_CLASSES and conf >= MIN_VEHICLE_CONF:
            vehicles.append(clean)
    return persons, vehicles


def match_riders(persons, vehicles):
    pairs = []
    used_persons = set()
    for vehicle_idx, vehicle in enumerate(vehicles):
        vb = vehicle["bbox"]
        vcenter = bbox_center(vb)
        best = None
        for person_idx, person in enumerate(persons):
            if person_idx in used_persons:
                continue
            pb = person["bbox"]
            pc = bbox_center(pb)
            d = dist(pc, vcenter)
            inter = intersection_area(pb, vb)
            overlap = inter / max(1, min(bbox_area(pb), bbox_area(vb)))
            vertical_ok = pc[1] <= vb[3] + 100
            close_ok = d <= MAX_CENTER_DISTANCE
            overlap_ok = overlap >= MIN_OVERLAP_RATIO
            if not vertical_ok or not (close_ok or overlap_ok):
                continue
            score = overlap * 2.0 + max(0, 1 - d / max(1, MAX_CENTER_DISTANCE))
            if best is None or score > best["match_score"]:
                best = {
                    "person_index": person_idx,
                    "vehicle_index": vehicle_idx,
                    "person": person,
                    "vehicle": vehicle,
                    "distance": round(d, 2),
                    "overlap": round(overlap, 4),
                    "match_score": round(score, 4),
                }
        if best is not None:
            used_persons.add(best["person_index"])
            pairs.append(best)
    return pairs




In [7]:
# Cell 7. SegFormer 로드: CUDA면 b5, CPU면 b0 fallback
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEG_MODEL_ID = SEG_MODEL_ID_GPU if DEVICE.type == "cuda" else SEG_MODEL_ID_CPU
print("DEVICE:", DEVICE)
print("SEG_MODEL_ID:", SEG_MODEL_ID)

if DEVICE.type != "cuda":
    print("경고: CUDA가 잡히지 않아 CPU로 실행합니다. CPU에서는 b0를 사용합니다.")

seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID)
seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID)
seg_model.to(DEVICE)
seg_model.eval()
print("SegFormer loaded")




DEVICE: cuda
SEG_MODEL_ID: nvidia/segformer-b5-finetuned-cityscapes-1024-1024


Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

SegFormer loaded


In [8]:
# Cell 8. 헬멧 모델 연결: 로컬 YOLO 또는 Roboflow hosted
rf_client = None
rf_hosted_ready = False

if USE_ROBOFLOW_HOSTED_INFERENCE:
    if not ROBOFLOW_API_KEY:
        print("ROBOFLOW_API_KEY가 없어 hosted inference를 사용할 수 없습니다.")
    else:
        try:
            from inference_sdk import InferenceHTTPClient
            rf_client = InferenceHTTPClient(api_url="https://detect.roboflow.com", api_key=ROBOFLOW_API_KEY)
            rf_hosted_ready = True
            print("Roboflow hosted inference ready:", ROBOFLOW_MODEL_ID)
        except Exception as e:
            print("Roboflow hosted inference init failed:", repr(e))
else:
    print("USE_ROBOFLOW_HOSTED_INFERENCE=False")

helmet_model = None
helmet_model_loaded = False
helmet_backend = "none"

if HELMET_MODEL_PATH is None:
    print("HELMET_MODEL_PATH is None -> 헬멧 모델 미연결. helmet_status는 unknown이 됩니다.")
elif YOLO is None:
    print("ultralytics가 없어 로컬 YOLO 헬멧 모델을 로드할 수 없음")
else:
    helmet_path = Path(HELMET_MODEL_PATH)
    if not helmet_path.exists():
        print("helmet model file not found:", helmet_path)
    else:
        helmet_model = YOLO(str(helmet_path))
        try:
            helmet_model.to(str(DEVICE))
        except Exception as e:
            print("helmet_model.to failed, continue:", repr(e))
        helmet_model_loaded = True
        helmet_backend = "local_yolo"
        print("helmet model loaded:", helmet_path)
        try:
            print("helmet model names:", helmet_model.names)
        except Exception:
            pass

if rf_hosted_ready:
    helmet_backend = "roboflow_hosted"

print("helmet_backend:", helmet_backend)




USE_ROBOFLOW_HOSTED_INFERENCE=False
HELMET_MODEL_PATH is None -> 헬멧 모델 미연결. helmet_status는 unknown이 됩니다.
helmet_backend: none


In [9]:
# Cell 9. SegFormer 기반 인도/차도 판정 함수
import numpy as np
from PIL import Image


def read_frame_from_cap(cap, frame_id: int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
    ret, frame_bgr = cap.read()
    if not ret:
        return None
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


def segment_frame(image_rgb: np.ndarray):
    pil_image = Image.fromarray(image_rgb)
    inputs = seg_processor(images=pil_image, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        if DEVICE.type == "cuda" and USE_AMP_ON_CUDA:
            with torch.cuda.amp.autocast():
                outputs = seg_model(**inputs)
        else:
            outputs = seg_model(**inputs)

    logits = outputs.logits
    upsampled_logits = torch.nn.functional.interpolate(
        logits,
        size=image_rgb.shape[:2],
        mode="bilinear",
        align_corners=False,
    )
    pred = upsampled_logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
    return pred


def make_class_masks(pred):
    road = (pred == ROAD_ID).astype(np.uint8)
    sidewalk = (pred == SIDEWALK_ID).astype(np.uint8)
    if DILATE_KERNEL_SIZE and DILATE_KERNEL_SIZE > 1:
        k = np.ones((DILATE_KERNEL_SIZE, DILATE_KERNEL_SIZE), np.uint8)
        road_d = cv2.dilate(road, k, iterations=1).astype(bool)
        sidewalk_d = cv2.dilate(sidewalk, k, iterations=1).astype(bool)
    else:
        road_d = road.astype(bool)
        sidewalk_d = sidewalk.astype(bool)
    return road.astype(bool), sidewalk.astype(bool), road_d, sidewalk_d


def safe_crop_mask(mask, x1, y1, x2, y2):
    h, w = mask.shape[:2]
    x1 = max(0, min(w, int(round(x1))))
    x2 = max(0, min(w, int(round(x2))))
    y1 = max(0, min(h, int(round(y1))))
    y2 = max(0, min(h, int(round(y2))))
    if x1 >= x2 or y1 >= y2:
        return None
    return mask[y1:y2, x1:x2]


def mask_patch_ratio(mask, center_xy, radius=MASK_PATCH_RADIUS):
    h, w = mask.shape[:2]
    x, y = center_xy
    x = int(round(x)); y = int(round(y))
    x1 = max(0, x - radius); x2 = min(w, x + radius + 1)
    y1 = max(0, y - radius); y2 = min(h, y + radius + 1)
    if x1 >= x2 or y1 >= y2:
        return 0.0
    return float(mask[y1:y2, x1:x2].mean())


def clip_box(box, w, h):
    x1, y1, x2, y2 = [float(v) for v in box]
    return [
        max(0, min(w - 1, x1)),
        max(0, min(h - 1, y1)),
        max(0, min(w - 1, x2)),
        max(0, min(h - 1, y2)),
    ]


def expand_box(box, w, h, scale_x=0.25, scale_y=0.10):
    x1, y1, x2, y2 = [float(v) for v in box]
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    return clip_box([x1 - bw * scale_x, y1 - bh * scale_y, x2 + bw * scale_x, y2 + bh * scale_y], w, h)


def union_box(box1, box2, w, h):
    x1 = min(float(box1[0]), float(box2[0]))
    y1 = min(float(box1[1]), float(box2[1]))
    x2 = max(float(box1[2]), float(box2[2]))
    y2 = max(float(box1[3]), float(box2[3]))
    return clip_box([x1, y1, x2, y2], w, h)


def bottom_band_from_box(box, x_margin=0.08, y1_ratio=0.70, y2_ratio=1.00):
    x1, y1, x2, y2 = [float(v) for v in box]
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    mx = bw * x_margin
    return x1 - mx, y1 + bh * y1_ratio, x2 + mx, y1 + bh * y2_ratio


def support_points_from_box(box):
    x1, y1, x2, y2 = [float(v) for v in box]
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    xs = [x1 + bw * r for r in [0.10, 0.22, 0.35, 0.50, 0.65, 0.78, 0.90]]
    ys = [y1 + bh * r for r in [0.72, 0.82, 0.92, 0.99]]
    return [(x, y) for y in ys for x in xs]


def score_surface_from_box(road_d, sidewalk_d, box, *, label="box"):
    h, w = road_d.shape[:2]
    x1, y1, x2, y2 = clip_box(box, w, h)
    if x2 <= x1 or y2 <= y1:
        return 0.0, 0.0, f"{label}: invalid_box"

    bx1, by1, bx2, by2 = bottom_band_from_box([x1, y1, x2, y2], x_margin=BOTTOM_BAND_X_MARGIN)
    road_band = safe_crop_mask(road_d, bx1, by1, bx2, by2)
    sidewalk_band = safe_crop_mask(sidewalk_d, bx1, by1, bx2, by2)
    road_band_ratio = float(road_band.mean()) if road_band is not None else 0.0
    sidewalk_band_ratio = float(sidewalk_band.mean()) if sidewalk_band is not None else 0.0

    pts = support_points_from_box([x1, y1, x2, y2])
    road_point_score = float(np.mean([mask_patch_ratio(road_d, p) for p in pts])) if pts else 0.0
    sidewalk_point_score = float(np.mean([mask_patch_ratio(sidewalk_d, p) for p in pts])) if pts else 0.0

    road_score = 0.60 * road_band_ratio + 0.40 * road_point_score
    sidewalk_score = 0.60 * sidewalk_band_ratio + 0.40 * sidewalk_point_score

    reason = (
        f"{label}: band road={road_band_ratio:.3f}, sidewalk={sidewalk_band_ratio:.3f}; "
        f"points road={road_point_score:.3f}, sidewalk={sidewalk_point_score:.3f}; "
        f"score road={road_score:.3f}, sidewalk={sidewalk_score:.3f}"
    )
    return road_score, sidewalk_score, reason


def judge_sidewalk_v2(pred, person_bbox, vehicle_bbox):
    road, sidewalk, road_d, sidewalk_d = make_class_masks(pred)
    h, w = pred.shape[:2]

    vehicle_expanded = expand_box(vehicle_bbox, w, h, scale_x=0.30, scale_y=0.10)
    pv_union = union_box(person_bbox, vehicle_bbox, w, h)
    union_expanded = expand_box(pv_union, w, h, scale_x=0.15, scale_y=0.05)

    vr, vs, vehicle_reason = score_surface_from_box(road_d, sidewalk_d, vehicle_expanded, label="vehicle_expanded")
    ur, us, union_reason = score_surface_from_box(road_d, sidewalk_d, union_expanded, label="person_vehicle_union")

    road_score = max(vr, ur * 0.85)
    sidewalk_score = max(vs, us)

    reason = (
        f"version={SIDEWALK_JUDGE_VERSION}; final road={road_score:.3f}, sidewalk={sidewalk_score:.3f}; "
        f"{vehicle_reason}; {union_reason}"
    )

    if sidewalk_score >= 0.18 and sidewalk_score >= road_score * 0.65:
        return "sidewalk_riding", "strong_sidewalk_surface_v2; " + reason, road_score, sidewalk_score
    if sidewalk_score >= 0.10 and sidewalk_score >= road_score * 0.45:
        return "sidewalk_candidate", "weak_sidewalk_surface_v2; " + reason, road_score, sidewalk_score
    if sidewalk_score > 0.05 and sidewalk_score > road_score:
        return "sidewalk_candidate", "sidewalk_more_than_road_v2; " + reason, road_score, sidewalk_score
    if road_score >= 0.20 and road_score > sidewalk_score * 1.40:
        return "road_riding", "road_surface_v2; " + reason, road_score, sidewalk_score
    return "unknown", "low_surface_confidence_v2; " + reason, road_score, sidewalk_score




In [10]:
# Cell 10. 헬멧 판정 함수
saved_crop_count = 0


def crop_head_and_upper_body(image_rgb: np.ndarray, person_bbox, mode="head"):
    if image_rgb is None:
        return None
    h_img, w_img = image_rgb.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in person_bbox]
    bw = x2 - x1
    bh = y2 - y1
    if bw <= 0 or bh <= 0:
        return None

    if mode == "head":
        cy2 = y1 + int(bh * 0.45)
        mx = int(bw * 0.35)
        my = int(bh * 0.12)
    else:
        cy2 = y1 + int(bh * 0.62)
        mx = int(bw * 0.25)
        my = int(bh * 0.08)

    cx1 = max(0, x1 - mx)
    cy1 = max(0, y1 - my)
    cx2 = min(w_img, x2 + mx)
    cy2 = min(h_img, cy2 + my)
    if cx1 >= cx2 or cy1 >= cy2:
        return None
    return image_rgb[cy1:cy2, cx1:cx2]


def normalize_helmet_label(label: str):
    label = str(label).lower().strip().replace("-", "_").replace(" ", "_")
    no_helmet_labels = {"no_helmet", "nohelmet", "without_helmet", "withouthelmet", "bare_head", "barehead", "rider_without_helmet", "without", "not_helmet"}
    helmet_labels = {"helmet", "with_helmet", "withhelmet", "hardhat", "hard_hat", "rider_with_helmet", "with"}
    if label in no_helmet_labels or "without" in label or "no_helmet" in label or "nohelmet" in label:
        return "no_helmet"
    if label in helmet_labels or "with_helmet" in label or "withhelmet" in label:
        return "helmet"
    return label


def save_debug_crop(crop, video_name, frame_id, pair_idx, mode):
    global saved_crop_count
    if not SAVE_HELMET_CROPS or crop is None or saved_crop_count >= MAX_SAVED_HELMET_CROPS:
        return None
    stem = Path(video_name).stem
    path = HELMET_CROP_DIR / f"{stem}_f{frame_id}_p{pair_idx}_{mode}.jpg"
    cv2.imwrite(str(path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
    saved_crop_count += 1
    return str(path)


def predict_helmet_local_yolo(crop):
    try:
        pred = helmet_model.predict(crop, verbose=False, conf=HELMET_CONF, device=0 if DEVICE.type == "cuda" else "cpu")
    except TypeError:
        pred = helmet_model.predict(crop, verbose=False, conf=HELMET_CONF)

    detections = []
    for res in pred:
        names = getattr(res, "names", getattr(helmet_model, "names", {}))
        boxes = getattr(res, "boxes", None)
        if boxes is None:
            continue
        for b in boxes:
            cls_id = int(b.cls[0].item())
            conf = float(b.conf[0].item())
            label = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
            norm = normalize_helmet_label(label)
            xyxy = [int(round(v)) for v in b.xyxy[0].detach().cpu().tolist()]
            detections.append({"label": label, "normalized": norm, "confidence": round(conf, 4), "bbox_crop": xyxy})
    return detections


def predict_helmet_roboflow(crop):
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        cv2.imwrite(tmp_path, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        response = rf_client.infer(tmp_path, model_id=ROBOFLOW_MODEL_ID)
    finally:
        try:
            os.remove(tmp_path)
        except Exception:
            pass

    detections = []
    for p in response.get("predictions", []):
        label = p.get("class", "")
        conf = float(p.get("confidence", 0.0))
        norm = normalize_helmet_label(label)
        x = float(p.get("x", 0)); y = float(p.get("y", 0)); w = float(p.get("width", 0)); h = float(p.get("height", 0))
        xyxy = [int(x - w/2), int(y - h/2), int(x + w/2), int(y + h/2)]
        detections.append({"label": label, "normalized": norm, "confidence": round(conf, 4), "bbox_crop": xyxy})
    return detections


def decide_helmet_from_detections(detections):
    if not detections:
        return "unknown", "helmet model returned no detection", []
    recognized = [d for d in detections if d.get("normalized") in {"helmet", "no_helmet"}]
    if not recognized:
        best = max(detections, key=lambda x: x["confidence"])
        return "unknown", f"helmet model labels not recognized: {best['label']}", detections

    no_hits = [d for d in recognized if d["normalized"] == "no_helmet"]
    hel_hits = [d for d in recognized if d["normalized"] == "helmet"]
    best_no = max(no_hits, key=lambda x: x["confidence"], default=None)
    best_hel = max(hel_hits, key=lambda x: x["confidence"], default=None)

    if best_no and (best_hel is None or best_no["confidence"] >= best_hel["confidence"] - 0.05):
        return "no_helmet", f"detected no_helmet conf={best_no['confidence']}", detections
    if best_hel:
        return "helmet", f"detected helmet conf={best_hel['confidence']}", detections
    return "unknown", "no recognized helmet/no_helmet detection", detections


def judge_helmet(image_rgb, person_bbox, video_name=None, frame_id=None, pair_idx=0):
    if helmet_backend == "none":
        return "unknown", "helmet model is not connected. Set HELMET_MODEL_PATH or Roboflow API.", []

    all_detections = []
    crop_paths = []
    for mode in ["head", "upper"]:
        crop = crop_head_and_upper_body(image_rgb, person_bbox, mode=mode)
        if crop is None:
            continue
        crop_path = save_debug_crop(crop, video_name or "video", frame_id or -1, pair_idx, mode)
        if crop_path:
            crop_paths.append(crop_path)

        try:
            if helmet_backend == "local_yolo":
                dets = predict_helmet_local_yolo(crop)
            elif helmet_backend == "roboflow_hosted":
                dets = predict_helmet_roboflow(crop)
            else:
                dets = []
        except Exception as e:
            return "unknown", f"helmet predict failed on {mode}: {repr(e)}", []

        for d in dets:
            d["crop_mode"] = mode
        all_detections.extend(dets)

        status, reason, _ = decide_helmet_from_detections(dets)
        if status in {"helmet", "no_helmet"}:
            if crop_paths:
                reason += f"; crop_saved={crop_paths[-1]}"
            return status, reason, all_detections

    status, reason, detections = decide_helmet_from_detections(all_detections)
    if crop_paths:
        reason += f"; crops_saved={len(crop_paths)}"
    return status, reason, detections




In [11]:
# Cell 11. cf6 병렬 판정 실행: 이륜차/인도주행/헬멧
cf6_results = []
cf6_failed_videos = []
start_time_all = time.time()

for json_idx, json_path in enumerate(json_files, start=1):
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    video_name = data.get("video_name")
    if not video_name:
        print("skip: no video_name", json_path.name)
        continue

    try:
        video_path = find_video_path(video_name)
    except Exception as e:
        cf6_failed_videos.append({"video_name": video_name, "reason": repr(e)})
        print("video not found:", video_name, repr(e))
        continue

    frames = data.get("frames", [])
    if MAX_FRAMES_PER_VIDEO is not None:
        frames = frames[:MAX_FRAMES_PER_VIDEO]

    print(f"\n[cf6 {json_idx}/{len(json_files)}] start {video_name}, frames={len(frames)}, device={DEVICE}, helmet_backend={helmet_backend}")
    t0 = time.time()

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cf6_failed_videos.append({"video_name": video_name, "reason": "cv2.VideoCapture failed"})
        print("video open failed:", video_path)
        continue

    video_pair_count = 0
    processed_pair_frames = 0

    for frame_idx, frame in enumerate(frames, start=1):
        frame_id = int(frame.get("frame_id", -1))
        persons, vehicles = extract_objects(frame)
        if not persons or not vehicles:
            continue

        pairs = match_riders(persons, vehicles)
        if not pairs:
            continue

        image_rgb = read_frame_from_cap(cap, frame_id)
        if image_rgb is None:
            print("frame read failed:", video_name, frame_id)
            continue

        try:
            pred = segment_frame(image_rgb)
        except RuntimeError as e:
            if DEVICE.type == "cuda" and "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                print("CUDA OOM at", video_name, frame_id, "-> skip frame")
                continue
            print("segmentation failed:", video_name, frame_id, repr(e))
            continue
        except Exception as e:
            print("segmentation failed:", video_name, frame_id, repr(e))
            continue

        processed_pair_frames += 1
        video_pair_count += len(pairs)

        for pair_idx, pair in enumerate(pairs):
            person = pair["person"]
            vehicle = pair["vehicle"]

            sidewalk_status, sidewalk_reason, road_score, sidewalk_score = judge_sidewalk_v2(
                pred,
                person_bbox=person["bbox"],
                vehicle_bbox=vehicle["bbox"],
            )
            helmet_status, helmet_reason, helmet_detections = judge_helmet(
                image_rgb, person["bbox"], video_name=video_name, frame_id=frame_id, pair_idx=pair_idx
            )

            anomaly_types = []
            if sidewalk_status in {"sidewalk_riding", "sidewalk_candidate"}:
                anomaly_types.append(sidewalk_status)
            if helmet_status == "no_helmet":
                anomaly_types.append("no_helmet")

            if sidewalk_status == "sidewalk_riding":
                loc_text = "인도 주행 의심"
            elif sidewalk_status == "sidewalk_candidate":
                loc_text = "인도 주행 후보"
            elif sidewalk_status == "road_riding":
                loc_text = "차도 주행"
            else:
                loc_text = "주행 위치 불확실"

            if helmet_status == "no_helmet":
                helmet_text = "헬멧 미착용 의심"
            elif helmet_status == "helmet":
                helmet_text = "헬멧 착용"
            else:
                helmet_text = "헬멧 판정 불확실"

            cf6_results.append({
                "video_name": video_name,
                "json_file": json_path.name,
                "video_path": str(video_path),
                "frame_id": frame_id,
                "pair_index": pair_idx,
                "person": person,
                "vehicle": vehicle,
                "match": {
                    "score": pair["match_score"],
                    "overlap": pair["overlap"],
                    "distance": pair["distance"],
                },
                "sidewalk_status": sidewalk_status,
                "sidewalk_reason": sidewalk_reason,
                "road_score": round(road_score, 4),
                "sidewalk_score": round(sidewalk_score, 4),
                "helmet_status": helmet_status,
                "helmet_reason": helmet_reason,
                "helmet_detections": helmet_detections,
                "anomaly_types": anomaly_types,
                "description": f"{loc_text}, {helmet_text}",
            })

        if processed_pair_frames % 5 == 0:
            print(
                f"  pair_frames={processed_pair_frames}, frame_id={frame_id}, pairs={video_pair_count}, "
                f"total_cf6_results={len(cf6_results)}, elapsed={round(time.time() - t0, 1)}s"
            )

        del image_rgb, pred

    cap.release()

    print(
        f"done {video_name}: pair_frames={processed_pair_frames}, pairs={video_pair_count}, "
        f"total_cf6_results={len(cf6_results)}, elapsed={round(time.time() - t0, 1)}s"
    )

    if json_idx % SAVE_PARTIAL_EVERY_N_VIDEOS == 0:
        partial_path = OT_CF6_DIR / "frame_priority_results_partial.json"
        with open(partial_path, "w", encoding="utf-8") as f:
            json.dump({"results": cf6_results, "failed_videos": cf6_failed_videos}, f, ensure_ascii=False, indent=2)
        print("partial saved:", partial_path)

    if DEVICE.type == "cuda" and json_idx % CLEAR_CUDA_CACHE_EVERY_N_VIDEOS == 0:
        torch.cuda.empty_cache()
        gc.collect()

print("\ncf6 processed results:", len(cf6_results))
print("cf6 failed videos:", len(cf6_failed_videos))
print("sidewalk status:", Counter(r["sidewalk_status"] for r in cf6_results))
print("helmet status:", Counter(r["helmet_status"] for r in cf6_results))
print("total elapsed sec:", round(time.time() - start_time_all, 1))





[cf6 1/22] start lo_e1281_c1.mp4, frames=61, device=cuda, helmet_backend=none


C:\Users\우성한\AppData\Local\Temp\ipykernel_28196\2903012533.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


done lo_e1281_c1.mp4: pair_frames=4, pairs=4, total_cf6_results=4, elapsed=3.3s

[cf6 2/22] start lo_e1281_c2.mp4, frames=62, device=cuda, helmet_backend=none
done lo_e1281_c2.mp4: pair_frames=4, pairs=4, total_cf6_results=8, elapsed=1.6s

[cf6 3/22] start fp_e1520_c1.mp4, frames=62, device=cuda, helmet_backend=none
done fp_e1520_c1.mp4: pair_frames=0, pairs=0, total_cf6_results=8, elapsed=0.0s

[cf6 4/22] start fp_e1520_c2.mp4, frames=62, device=cuda, helmet_backend=none
done fp_e1520_c2.mp4: pair_frames=0, pairs=0, total_cf6_results=8, elapsed=0.0s

[cf6 5/22] start ph_e2063_c1.mp4, frames=63, device=cuda, helmet_backend=none
done ph_e2063_c1.mp4: pair_frames=0, pairs=0, total_cf6_results=8, elapsed=0.1s
partial saved: C:\Users\우성한\outputs\prediction\ot3\cf6_parallel\frame_priority_results_partial.json

[cf6 6/22] start ph_e2063_c2.mp4, frames=62, device=cuda, helmet_backend=none
done ph_e2063_c2.mp4: pair_frames=1, pairs=1, total_cf6_results=9, elapsed=0.5s

[cf6 7/22] start ms_e350

In [12]:
# Cell 12. cf6 결과 저장 및 구간 병합

def merge_segments_light(items, status_key, target_statuses, max_gap=10):
    if isinstance(target_statuses, str):
        target_statuses = {target_statuses}
    else:
        target_statuses = set(target_statuses)

    grouped = defaultdict(list)
    for item in items:
        if item.get(status_key) in target_statuses:
            grouped[item["video_name"]].append(int(item["frame_id"]))

    segments = []
    for video_name, fids in grouped.items():
        fids = sorted(set(fids))
        if not fids:
            continue
        start = fids[0]
        end = fids[0]
        count = 1
        for fid in fids[1:]:
            if fid - end <= max_gap:
                end = fid
                count += 1
            else:
                segments.append({"video_name": video_name, "start_frame": start, "end_frame": end, "count": count})
                start = fid
                end = fid
                count = 1
        segments.append({"video_name": video_name, "start_frame": start, "end_frame": end, "count": count})
    return segments

sidewalk_segments = merge_segments_light(cf6_results, "sidewalk_status", {"sidewalk_riding", "sidewalk_candidate"})
no_helmet_segments = merge_segments_light(cf6_results, "helmet_status", "no_helmet")

cf6_segments = []
for s in sidewalk_segments:
    s = dict(s)
    s["event_type"] = "sidewalk_riding"
    s["description"] = "이륜차 인도 주행 또는 인도 주행 후보 구간"
    cf6_segments.append(s)
for s in no_helmet_segments:
    s = dict(s)
    s["event_type"] = "no_helmet"
    s["description"] = "이륜차 헬멧 미착용 의심 구간"
    cf6_segments.append(s)

cf6_output = {
    "meta": {
        "input_cf2_dir": str(CF2_DIR),
        "video_root": str(VIDEO_ROOT),
        "device": str(DEVICE),
        "seg_model_id": SEG_MODEL_ID,
        "helmet_backend": helmet_backend,
        "helmet_model_path": None if HELMET_MODEL_PATH is None else str(HELMET_MODEL_PATH),
        "road_id": ROAD_ID,
        "sidewalk_id": SIDEWALK_ID,
        "sidewalk_judge_version": SIDEWALK_JUDGE_VERSION,
    },
    "summary": {
        "processed_results": len(cf6_results),
        "failed_video_count": len(cf6_failed_videos),
        "sidewalk_status": dict(Counter(r["sidewalk_status"] for r in cf6_results)),
        "helmet_status": dict(Counter(r["helmet_status"] for r in cf6_results)),
        "anomaly_types": dict(Counter(t for r in cf6_results for t in r.get("anomaly_types", []))),
    },
    "failed_videos": cf6_failed_videos,
    "results": cf6_results,
}

cf6_frame_out = OT_CF6_DIR / "frame_parallel_results.json"
cf6_segment_out = OT_CF6_DIR / "segment_parallel_results.json"

with open(cf6_frame_out, "w", encoding="utf-8") as f:
    json.dump(cf6_output, f, ensure_ascii=False, indent=2)
with open(cf6_segment_out, "w", encoding="utf-8") as f:
    json.dump({"meta": cf6_output["meta"], "segments": cf6_segments}, f, ensure_ascii=False, indent=2)

print("saved:", cf6_frame_out)
print("saved:", cf6_segment_out)
print("cf6 summary:", json.dumps(cf6_output["summary"], ensure_ascii=False, indent=2))




saved: C:\Users\우성한\outputs\prediction\ot3\cf6_parallel\frame_parallel_results.json
saved: C:\Users\우성한\outputs\prediction\ot3\cf6_parallel\segment_parallel_results.json
cf6 summary: {
  "processed_results": 80,
  "failed_video_count": 0,
  "sidewalk_status": {
    "road_riding": 49,
    "sidewalk_riding": 30,
    "unknown": 1
  },
  "helmet_status": {
    "unknown": 80
  },
  "anomaly_types": {
    "sidewalk_riding": 30
  }
}


In [13]:
# Cell 13. bd4 행동 판정 함수: fight / stalking / other

def extract_persons_for_bd4(frame):
    persons = []
    for obj in frame.get("objects", []):
        if obj.get("class") == "person":
            bbox = obj.get("bbox")
            if bbox and len(bbox) == 4:
                persons.append({
                    "bbox": bbox,
                    "center": bbox_center(bbox),
                    "area": bbox_area(bbox),
                    "confidence": obj.get("confidence", 0),
                })
    return persons


def classify_bd4_situation(data):
    frames = data.get("frames", [])
    if not frames:
        return {
            "label": "other",
            "reason": "분석 가능한 프레임 정보가 없음",
            "features": {},
            "scores": {"fight_score": 0, "stalking_score": 0},
        }

    person_frame_count = 0
    multi_person_frame_count = 0
    close_pair_count = 0
    movement_events = 0
    follow_like_events = 0
    prev_persons = []

    for frame in frames:
        persons = extract_persons_for_bd4(frame)

        if len(persons) > 0:
            person_frame_count += 1

        if len(persons) >= 2:
            multi_person_frame_count += 1
            centers = [p["center"] for p in persons]
            min_d = None
            for i in range(len(centers)):
                for j in range(i + 1, len(centers)):
                    d = dist(centers[i], centers[j])
                    if min_d is None or d < min_d:
                        min_d = d
            if min_d is not None and min_d < 180:
                close_pair_count += 1

        if prev_persons and persons:
            curr_centers = [p["center"] for p in persons]
            prev_centers = [p["center"] for p in prev_persons]
            for pc in prev_centers:
                nearest_d = min(dist(pc, cc) for cc in curr_centers)
                if nearest_d > 60:
                    movement_events += 1

            if len(prev_persons) >= 2 and len(persons) >= 2:
                prev_centers_sorted = sorted(prev_centers, key=lambda x: x[0])
                curr_centers_sorted = sorted(curr_centers, key=lambda x: x[0])
                prev_gap = dist(prev_centers_sorted[0], prev_centers_sorted[1])
                curr_gap = dist(curr_centers_sorted[0], curr_centers_sorted[1])
                if abs(curr_gap - prev_gap) < 80:
                    follow_like_events += 1

        prev_persons = persons

    total_frames = len(frames)
    person_ratio = person_frame_count / total_frames
    multi_person_ratio = multi_person_frame_count / total_frames
    close_pair_ratio = close_pair_count / total_frames
    movement_ratio = movement_events / total_frames
    follow_like_ratio = follow_like_events / total_frames

    fight_score = 0
    stalking_score = 0

    if multi_person_ratio >= 0.25:
        fight_score += 1
    if close_pair_ratio >= 0.15:
        fight_score += 1
    if movement_ratio >= 0.25:
        fight_score += 1

    if person_ratio >= 0.35:
        stalking_score += 1
    if follow_like_ratio >= 0.10:
        stalking_score += 1
    if movement_ratio >= 0.12:
        stalking_score += 1
    if close_pair_ratio < 0.20:
        stalking_score += 1

    if fight_score >= 2:
        label = "fight"
        reason = "여러 사람이 함께 등장하고, 인물 간 거리가 가까우며, 프레임 간 움직임 변화가 커서 싸움 후보로 판단"
    elif stalking_score >= 2:
        label = "stalking"
        reason = "사람이 지속적으로 등장하고, 이동 흐름 또는 따라가는 형태의 움직임이 관찰되어 스토킹 후보로 판단"
    else:
        label = "other"
        reason = "fight 또는 stalking으로 볼 만큼의 다인 근접 상호작용이나 추적형 움직임 특징이 부족하여 기타 상황으로 판단"

    return {
        "label": label,
        "reason": reason,
        "features": {
            "total_frames": total_frames,
            "person_frame_count": person_frame_count,
            "multi_person_frame_count": multi_person_frame_count,
            "close_pair_count": close_pair_count,
            "movement_events": movement_events,
            "follow_like_events": follow_like_events,
            "person_ratio": round(person_ratio, 4),
            "multi_person_ratio": round(multi_person_ratio, 4),
            "close_pair_ratio": round(close_pair_ratio, 4),
            "movement_ratio": round(movement_ratio, 4),
            "follow_like_ratio": round(follow_like_ratio, 4),
        },
        "scores": {"fight_score": fight_score, "stalking_score": stalking_score},
    }




In [14]:
# Cell 14. bd4 행동 판정 실행 및 저장
bd4_results = []
bd4_failed = []

for json_path in tqdm(json_files):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        pred = classify_bd4_situation(data)
        item = {
            "cf2_json": str(json_path),
            "video_name": data.get("video_name"),
            "video_path": data.get("video_path"),
            "relative_video_path": data.get("relative_video_path"),
            "label": pred["label"],
            "reason": pred["reason"],
            "features": pred.get("features", {}),
            "scores": pred.get("scores", {}),
        }
        bd4_results.append(item)

        out_path = OT_BD4_DIR / json_path.name
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(item, f, ensure_ascii=False, indent=2)
    except Exception as e:
        bd4_failed.append({"json": str(json_path), "error": repr(e)})

bd4_summary = {
    "task": "ot3_bd4_fight_stalking_other_classification",
    "video_root": str(VIDEO_ROOT),
    "cf2_dir": str(CF2_DIR),
    "output_dir": str(OT_BD4_DIR),
    "total_json": len(json_files),
    "success_count": len(bd4_results),
    "failed_count": len(bd4_failed),
    "fight_count": sum(1 for r in bd4_results if r["label"] == "fight"),
    "stalking_count": sum(1 for r in bd4_results if r["label"] == "stalking"),
    "other_count": sum(1 for r in bd4_results if r["label"] == "other"),
    "results": bd4_results,
    "failed": bd4_failed,
}

bd4_summary_path = OT_BD4_DIR / "bd4_parallel_summary.json"
with open(bd4_summary_path, "w", encoding="utf-8") as f:
    json.dump(bd4_summary, f, ensure_ascii=False, indent=2)

print("\n================================")
print("bd4 행동 판정 완료")
print("전체 json:", bd4_summary["total_json"])
print("성공:", bd4_summary["success_count"])
print("실패:", bd4_summary["failed_count"])
print("fight:", bd4_summary["fight_count"])
print("stalking:", bd4_summary["stalking_count"])
print("other:", bd4_summary["other_count"])
print("summary 저장:", bd4_summary_path)




100%|█████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 386.23it/s]


bd4 행동 판정 완료
전체 json: 22
성공: 22
실패: 0
fight: 7
stalking: 5
other: 10
summary 저장: C:\Users\우성한\outputs\prediction\ot3\bd4_parallel\bd4_parallel_summary.json


In [15]:
# Cell 15. OT3 최종 통합: 우선순위 없이 동시 판정 결과 보존

def cf6_video_parallel_summary(cf6_results):
    grouped = defaultdict(list)
    for r in cf6_results:
        grouped[r["video_name"]].append(r)

    summary = {}
    for video_name, items in grouped.items():
        sidewalk_strong_hits = [r for r in items if r.get("sidewalk_status") == "sidewalk_riding"]
        sidewalk_candidate_hits = [r for r in items if r.get("sidewalk_status") == "sidewalk_candidate"]
        sidewalk_hits = sidewalk_strong_hits + sidewalk_candidate_hits
        no_helmet_hits = [r for r in items if r.get("helmet_status") == "no_helmet"]
        helmet_hits = [r for r in items if r.get("helmet_status") == "helmet"]
        road_hits = [r for r in items if r.get("sidewalk_status") == "road_riding"]
        vehicle_hits = items

        cf6_tags = []
        if vehicle_hits:
            cf6_tags.append("two_wheeler_detected")
        if sidewalk_strong_hits:
            cf6_tags.append("two_wheeler_sidewalk_riding")
        elif sidewalk_candidate_hits:
            cf6_tags.append("two_wheeler_sidewalk_candidate")
        if no_helmet_hits:
            cf6_tags.append("two_wheeler_no_helmet")
        if road_hits and not sidewalk_hits:
            cf6_tags.append("two_wheeler_road_riding")
        if helmet_hits and not no_helmet_hits:
            cf6_tags.append("two_wheeler_helmet")

        if not cf6_tags:
            cf6_label = "no_two_wheeler_event"
            cf6_description = "cf6: 이륜차 관련 매칭 결과 없음"
        elif "two_wheeler_sidewalk_riding" in cf6_tags and "two_wheeler_no_helmet" in cf6_tags:
            cf6_label = "two_wheeler_sidewalk_and_no_helmet"
            cf6_description = "cf6: 이륜차 인도 주행 및 헬멧 미착용 의심"
        elif "two_wheeler_sidewalk_riding" in cf6_tags:
            cf6_label = "two_wheeler_sidewalk_riding"
            cf6_description = "cf6: 이륜차 인도 주행 의심"
        elif "two_wheeler_sidewalk_candidate" in cf6_tags:
            cf6_label = "two_wheeler_sidewalk_candidate"
            cf6_description = "cf6: 이륜차 인도 주행 후보"
        elif "two_wheeler_no_helmet" in cf6_tags:
            cf6_label = "two_wheeler_no_helmet"
            cf6_description = "cf6: 이륜차 헬멧 미착용 의심"
        elif "two_wheeler_road_riding" in cf6_tags:
            cf6_label = "two_wheeler_road_riding"
            cf6_description = "cf6: 이륜차 차도 주행"
        else:
            cf6_label = "two_wheeler_normal_or_uncertain"
            cf6_description = "cf6: 이륜차는 감지되었으나 이상 여부는 불확실"

        summary[video_name] = {
            "video_name": video_name,
            "cf6_label": cf6_label,
            "cf6_description": cf6_description,
            "cf6_tags": cf6_tags,
            "counts": {
                "total_pair_results": len(items),
                "sidewalk_strong_hits": len(sidewalk_strong_hits),
                "sidewalk_candidate_hits": len(sidewalk_candidate_hits),
                "sidewalk_hits": len(sidewalk_hits),
                "no_helmet_hits": len(no_helmet_hits),
                "helmet_hits": len(helmet_hits),
                "road_hits": len(road_hits),
            },
            "sample_hits": (sidewalk_hits + no_helmet_hits + road_hits + vehicle_hits)[:5],
        }
    return summary


cf6_video_summary = cf6_video_parallel_summary(cf6_results)
bd4_by_video = {r["video_name"]: r for r in bd4_results}
all_video_names = sorted(set([p.name for p in video_paths]) | set(bd4_by_video.keys()) | set(cf6_video_summary.keys()))

ot3_results = []
for video_name in all_video_names:
    cf6_item = cf6_video_summary.get(video_name)
    bd4_item = bd4_by_video.get(video_name)

    bd4_label = bd4_item.get("label", "other") if bd4_item else "other"
    bd4_tags = []
    if bd4_label in {"fight", "stalking", "other"}:
        bd4_tags.append(bd4_label)

    cf6_label = cf6_item.get("cf6_label", "no_two_wheeler_event") if cf6_item else "no_two_wheeler_event"
    cf6_tags = cf6_item.get("cf6_tags", []) if cf6_item else []

    combined_tags = []
    for tag in bd4_tags + cf6_tags:
        if tag not in combined_tags:
            combined_tags.append(tag)

    # 사람이 보기 쉬운 동시 판정 요약. 우선순위 판정이 아니라 표시용 문자열임.
    if cf6_label != "no_two_wheeler_event":
        parallel_label = f"{bd4_label} + {cf6_label}"
    else:
        parallel_label = bd4_label

    video_path = None
    relative_video_path = None
    if bd4_item:
        video_path = bd4_item.get("video_path")
        relative_video_path = bd4_item.get("relative_video_path")
    elif video_name in video_index:
        video_path = str(video_index[video_name])
        try:
            relative_video_path = str(video_index[video_name].relative_to(VIDEO_ROOT))
        except Exception:
            relative_video_path = None

    ot3_results.append({
        "video_name": video_name,
        "video_path": video_path,
        "relative_video_path": relative_video_path,
        "parallel_label": parallel_label,
        "combined_tags": combined_tags,
        "bd4_label": bd4_label,
        "bd4_action": bd4_item,
        "cf6_label": cf6_label,
        "cf6_two_wheeler": cf6_item,
        "description": {
            "bd4": bd4_item.get("reason") if bd4_item else "bd4 결과 없음. other로 처리",
            "cf6": cf6_item.get("cf6_description") if cf6_item else "cf6: 이륜차 관련 매칭 결과 없음",
        },
    })

ot3_summary = {
    "task": "ot3_parallel_bd4_and_cf6_classification",
    "video_root": str(VIDEO_ROOT),
    "output_dir": str(OT3_DIR),
    "rule": "No priority. bd4 fight/stalking/other and cf6 two-wheeler/sidewalk/no-helmet are stored simultaneously.",
    "summary": {
        "total_videos": len(ot3_results),
        "bd4_label": dict(Counter(r["bd4_label"] for r in ot3_results)),
        "cf6_label": dict(Counter(r["cf6_label"] for r in ot3_results)),
        "combined_tags": dict(Counter(tag for r in ot3_results for tag in r.get("combined_tags", []))),
        "cf6_frame_results": len(cf6_results),
        "bd4_video_results": len(bd4_results),
    },
    "results": ot3_results,
    "cf2_failed": failed_cf2,
    "cf6_failed": cf6_failed_videos,
    "bd4_failed": bd4_failed,
}

ot3_summary_path = OT3_DIR / "ot3_final_summary.json"
with open(ot3_summary_path, "w", encoding="utf-8") as f:
    json.dump(ot3_summary, f, ensure_ascii=False, indent=2)

print("\n================================")
print("OT3 동시 통합 완료")
print("================================")
print("total_videos:", ot3_summary["summary"]["total_videos"])
print("bd4_label:", ot3_summary["summary"]["bd4_label"])
print("cf6_label:", ot3_summary["summary"]["cf6_label"])
print("combined_tags:", ot3_summary["summary"]["combined_tags"])
print("summary 저장:", ot3_summary_path)





OT3 동시 통합 완료
total_videos: 22
bd4_label: {'other': 10, 'fight': 7, 'stalking': 5}
cf6_label: {'no_two_wheeler_event': 13, 'two_wheeler_road_riding': 7, 'two_wheeler_sidewalk_riding': 2}
combined_tags: {'other': 10, 'fight': 7, 'two_wheeler_detected': 9, 'two_wheeler_road_riding': 7, 'stalking': 5, 'two_wheeler_sidewalk_riding': 2}
summary 저장: C:\Users\우성한\outputs\prediction\ot3\ot3_final_summary.json


In [16]:
# Cell 16. 동시 판정 결과 출력: bd4와 cf6를 나란히 보기

print("\n" + "=" * 80)
print("BD4 행동 판정별 분포")
print("=" * 80)
for label in ["fight", "stalking", "other"]:
    rows = [r for r in ot3_results if r["bd4_label"] == label]
    print(f"{label}: {len(rows)}")

print("\n" + "=" * 80)
print("CF6 이륜차 판정별 분포")
print("=" * 80)
for label, count in sorted(Counter(r["cf6_label"] for r in ot3_results).items()):
    print(f"{label}: {count}")

print("\n" + "=" * 80)
print("영상별 동시 판정 결과")
print("=" * 80)
for r in ot3_results[:100]:
    print("video_name:", r.get("video_name"))
    print("relative_video_path:", r.get("relative_video_path"))
    print("parallel_label:", r.get("parallel_label"))
    print("combined_tags:", r.get("combined_tags"))
    print("bd4_label:", r.get("bd4_label"))
    if r.get("bd4_action"):
        print("bd4_scores:", r["bd4_action"].get("scores"))
    print("cf6_label:", r.get("cf6_label"))
    if r.get("cf6_two_wheeler"):
        print("cf6_counts:", r["cf6_two_wheeler"].get("counts"))
    print("-" * 80)





BD4 행동 판정별 분포
fight: 7
stalking: 5
other: 10

CF6 이륜차 판정별 분포
no_two_wheeler_event: 13
two_wheeler_road_riding: 7
two_wheeler_sidewalk_riding: 2

영상별 동시 판정 결과
video_name: ab_e0457_c1.mp4
relative_video_path: 침입\비정상적인 경로로의 침범\ab_e0457\ab_e0457_c1.mp4
parallel_label: other
combined_tags: ['other']
bd4_label: other
bd4_scores: {'fight_score': 0, 'stalking_score': 1}
cf6_label: no_two_wheeler_event
--------------------------------------------------------------------------------
video_name: ab_e0457_c2.mp4
relative_video_path: 침입\비정상적인 경로로의 침범\ab_e0457\ab_e0457_c2.mp4
parallel_label: other
combined_tags: ['other']
bd4_label: other
bd4_scores: {'fight_score': 0, 'stalking_score': 1}
cf6_label: no_two_wheeler_event
--------------------------------------------------------------------------------
video_name: bl_e0272_c1.mp4
relative_video_path: 침입\경계선을 통한 침입\bl_e0272\bl_e0272_c1.mp4
parallel_label: fight + two_wheeler_road_riding
combined_tags: ['fight', 'two_wheeler_detected', 'two_wheeler_roa

In [17]:
# Cell 17. 자연어 질의용 간단 검색 함수: 동시 태그 기반

def search_ot3_results(query: str, limit=20):
    q = query.lower()

    wants_sidewalk = any(k in query for k in ["인도", "보도", "sidewalk"])
    wants_candidate = any(k in query for k in ["후보", "candidate"])
    wants_no_helmet = any(k in query for k in ["헬멧 안", "헬멧 미착용", "노헬멧", "no helmet", "no_helmet"])
    wants_two_wheeler = any(k in query for k in ["이륜", "오토바이", "자전거", "bike", "motorcycle", "bicycle"])
    wants_fight = any(k in query for k in ["싸움", "폭행", "fight"])
    wants_stalking = any(k in query for k in ["스토킹", "따라", "stalking"])
    wants_other = any(k in query for k in ["기타", "other"])
    wants_overlap = any(k in query for k in ["겹", "동시", "같이", "함께", "overlap"])

    hits = []
    for r in ot3_results:
        tags = set(r.get("combined_tags", []))
        bd4_label = r.get("bd4_label", "")
        cf6_label = r.get("cf6_label", "")
        text = json.dumps(r, ensure_ascii=False).lower()

        conditions = []
        if wants_sidewalk:
            if wants_candidate:
                conditions.append("two_wheeler_sidewalk_candidate" in tags or "sidewalk_candidate" in cf6_label)
            else:
                conditions.append(any("sidewalk" in t for t in tags) or "sidewalk" in cf6_label)
        if wants_no_helmet:
            conditions.append("two_wheeler_no_helmet" in tags or "no_helmet" in cf6_label)
        if wants_two_wheeler:
            conditions.append("two_wheeler_detected" in tags or cf6_label != "no_two_wheeler_event")
        if wants_fight:
            conditions.append(bd4_label == "fight")
        if wants_stalking:
            conditions.append(bd4_label == "stalking")
        if wants_other:
            conditions.append(bd4_label == "other")
        if wants_overlap:
            # 행동 이상(fight/stalking)과 이륜차 계열이 동시에 있는 경우
            conditions.append(bd4_label in {"fight", "stalking"} and cf6_label != "no_two_wheeler_event")

        if conditions:
            ok = all(conditions)
        else:
            ok = q in text

        if ok:
            hits.append(r)

    print("QUERY:", query)
    print("hits:", len(hits))
    for h in hits[:limit]:
        print({
            "video_name": h.get("video_name"),
            "relative_video_path": h.get("relative_video_path"),
            "parallel_label": h.get("parallel_label"),
            "combined_tags": h.get("combined_tags"),
            "bd4_label": h.get("bd4_label"),
            "cf6_label": h.get("cf6_label"),
        })
    return hits

# 예시:
# search_ot3_results("싸움")
# search_ot3_results("스토킹")
# search_ot3_results("이륜차")
# search_ot3_results("싸움과 이륜차가 겹친 영상")
# search_ot3_results("인도 주행 이륜차")
# search_ot3_results("헬멧 미착용")
